# Tracker slot visibility over the match

This notebook reads the tactical ATD metadata and raw tracking file, plus the tactical homography and video dimensions. Each track slot is shown across the full video timeline in five states:

- **Gray** — no player position on the pitch (including frames where the slot is absent).
- **Red / blue** — home / away position projects inside the camera frame.
- **Light red / light blue** — home / away position is on the pitch but outside the camera view.

Camera visibility follows the match viewer's `visiblePitchPolygon` logic: the homography must be usable, the position must lie within normalized pitch bounds, and its projected image point must lie inside the video dimensions. Percentages use all video frames as the denominator, so halftime and pre/post-match frames count as not on pitch.

ATD player entries are **track slots**, not persistent person identities.

In [ ]:
%pip install opencv-python
import sys
import xml.etree.ElementTree as ET
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "starter" else NOTEBOOK_DIR
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from games import GAME, GAMES, asset_paths

paths = asset_paths(game=GAME, source="tactical")
N_FRAMES = GAMES[GAME]["n_video_frames"]
FPS = GAMES[GAME]["fps"]
PITCH_BOUNDS = (0.0, 0.0, 1.0, 1.0)
print("video frames:", N_FRAMES, "at", FPS, "fps")
print("ATD metadata:", paths["atd_meta"])
print("ATD raw data:", paths["atd_raw"])
print("homography:", paths["homography"])

## Read teams and tracker slots

The raw ATD stores player groups in the same slot order as the `Player` elements in the metadata. The `teamId` in the XML supplies the tracker-assigned side.

In [ ]:
def local_name(element):
    return element.tag.rsplit("}", 1)[-1]


metadata_root = ET.parse(paths["atd_meta"]).getroot()
teams = {}
slots = []
for element in metadata_root.iter():
    tag = local_name(element)
    if tag == "Team":
        team_id = element.get("id") or element.get("teamId")
        if team_id:
            team_name = next(
                ((child.text or "").strip() for child in element
                 if local_name(child) == "Name"),
                team_id,
            )
            teams[team_id] = team_name
    elif tag == "Player":
        player_id = element.get("id") or element.get("playerId")
        if player_id:
            player_name = next(
                ((child.text or "").strip() for child in element
                 if local_name(child) == "Name"),
                "Track " + player_id,
            )
            slots.append({
                "id": player_id,
                "name": player_name,
                "team_id": element.get("teamId", ""),
            })

if len(teams) != 2 or not slots:
    raise ValueError("Expected two teams and at least one player slot in the ATD metadata")
unknown_team_ids = sorted({slot["team_id"] for slot in slots} - set(teams))
if unknown_team_ids:
    raise ValueError("Player slots reference unknown teams: {}".format(unknown_team_ids))

team_ids = list(teams)
team_side = {team_ids[0]: "home", team_ids[1]: "away"}
for slot in slots:
    slot["side"] = team_side[slot["team_id"]]

print("home:", teams[team_ids[0]], "| away:", teams[team_ids[1]])
print("tracker slots:", len(slots))
pd.DataFrame(slots)[["id", "name", "side"]].head()

## Load camera geometry

Homography rows use zero-based video frame numbers, matching the ATD frame after subtracting one from its one-based `frame_id`. A position is camera-visible when the viewer's camera projection is defined and falls within the video image.

In [ ]:
video_path = paths["video"]
capture = cv2.VideoCapture(video_path)
if not capture.isOpened():
    raise OSError("Could not open tactical video: {}".format(video_path))
VIDEO_WIDTH = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
VIDEO_HEIGHT = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
VIDEO_FRAMES = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
capture.release()
if VIDEO_FRAMES != N_FRAMES:
    raise ValueError("Video has {} frames, expected {}".format(VIDEO_FRAMES, N_FRAMES))

homography_table = pd.read_csv(paths["homography"])
matrix_columns = ["h{}{}".format(row, col) for row in range(3) for col in range(3)]
required_columns = {"frame", "valid"} | set(matrix_columns)
missing_columns = required_columns - set(homography_table.columns)
if missing_columns:
    raise ValueError("Homography is missing columns: {}".format(sorted(missing_columns)))
homography_table = homography_table.sort_values("frame")
if not np.array_equal(homography_table["frame"].to_numpy(), np.arange(N_FRAMES)):
    raise ValueError("Homography rows do not match zero-based video frames")

homographies = homography_table[matrix_columns].to_numpy(dtype=np.float64).reshape(-1, 3, 3)
homography_valid = homography_table["valid"].to_numpy(dtype=bool)
print("video size:", VIDEO_WIDTH, "x", VIDEO_HEIGHT)
print("valid homographies:", int(homography_valid.sum()), "/", N_FRAMES)

## Classify every slot in every frame

State codes are `0 = not on pitch`, `1 = home on camera`, `2 = away on camera`, `3 = home off camera`, and `4 = away off camera`. A missing ATD position is gray. An on-pitch position is light red/blue if its frame's homography is invalid or it projects outside the image.

In [ ]:
NOT_ON_PITCH, HOME_VISIBLE, AWAY_VISIBLE, HOME_OFF_CAMERA, AWAY_OFF_CAMERA = range(5)
state_names = [
    "Not on pitch",
    "Home on camera",
    "Away on camera",
    "Home off camera",
    "Away off camera",
]
state_colors = ["#808080", "#eb4646", "#32aaeb", "#f3a0a0", "#9bd4ef"]
side_codes = np.asarray([
    (HOME_VISIBLE, HOME_OFF_CAMERA) if slot["side"] == "home"
    else (AWAY_VISIBLE, AWAY_OFF_CAMERA)
    for slot in slots
], dtype=np.int8)

states = np.zeros((N_FRAMES, len(slots)), dtype=np.uint8)
tracking_positions = np.full((N_FRAMES, len(slots), 2), np.nan, dtype=np.float32)
player_speeds = np.full((N_FRAMES, len(slots)), np.nan, dtype=np.float32)
with open(paths["atd_raw"], encoding="utf-8") as tracking_file:
    for line_number, line in enumerate(tracking_file, 1):
        frame_text, separator, remainder = line.rstrip("\r\n").partition(":")
        if not separator:
            raise ValueError("{}:{}: missing frame separator".format(
                paths["atd_raw"], line_number
            ))
        frame_id = int(frame_text)
        frame = frame_id - 1
        if frame != line_number - 1 or frame < 0 or frame >= N_FRAMES:
            raise ValueError("ATD frame ids are not contiguous 1..{} at line {}".format(
                N_FRAMES, line_number
            ))
        fields = remainder.split(":")
        if len(fields) != 2:
            raise ValueError("{}:{}: expected player and ball fields".format(
                paths["atd_raw"], line_number
            ))
        player_groups = fields[0].split(";")
        if len(player_groups) != len(slots):
            raise ValueError("{}:{}: expected {} player slots, found {}".format(
                paths["atd_raw"], line_number, len(slots), len(player_groups)
            ))

        positions = np.full((len(slots), 2), np.nan, dtype=np.float64)
        for slot_index, group in enumerate(player_groups):
            values = group.split(",")
            if len(values) < 2:
                raise ValueError("{}:{}: malformed player position".format(
                    paths["atd_raw"], line_number
                ))
            try:
                positions[slot_index] = (float(values[0]), float(values[1]))
            except ValueError:
                pass
            try:
                player_speeds[frame, slot_index] = float(values[2])
            except (IndexError, ValueError):
                pass

        tracking_positions[frame] = positions

        on_pitch = (
            np.isfinite(positions).all(axis=1)
            & (positions[:, 0] >= PITCH_BOUNDS[0])
            & (positions[:, 0] <= PITCH_BOUNDS[2])
            & (positions[:, 1] >= PITCH_BOUNDS[1])
            & (positions[:, 1] <= PITCH_BOUNDS[3])
        )
        if not on_pitch.any():
            continue

        # Default tracked, on-pitch players to off-camera; upgrade those in the camera view.
        states[frame, on_pitch] = side_codes[on_pitch, 1]
        matrix = homographies[frame]
        corners = np.asarray([
            [PITCH_BOUNDS[0], PITCH_BOUNDS[1]],
            [PITCH_BOUNDS[2], PITCH_BOUNDS[1]],
            [PITCH_BOUNDS[2], PITCH_BOUNDS[3]],
            [PITCH_BOUNDS[0], PITCH_BOUNDS[3]],
        ])
        corner_w = matrix[2, 0] * corners[:, 0] + matrix[2, 1] * corners[:, 1] + matrix[2, 2]
        projective_view_valid = (
            np.isfinite(matrix).all()
            and np.isfinite(corner_w).all()
            and np.all(np.abs(corner_w) >= 1e-8)
            and np.all(np.sign(corner_w) == np.sign(corner_w[0]))
        )
        if not homography_valid[frame] or not projective_view_valid:
            continue

        indices = np.flatnonzero(on_pitch)
        xy = positions[indices]
        homogeneous = np.column_stack([xy, np.ones(len(xy))]) @ matrix.T
        denominator = homogeneous[:, 2]
        projectable = np.isfinite(homogeneous).all(axis=1) & (np.abs(denominator) >= 1e-8)
        pixels = np.full((len(indices), 2), np.nan, dtype=np.float64)
        pixels[projectable] = homogeneous[projectable, :2] / denominator[projectable, None]
        in_camera = (
            projectable
            & (pixels[:, 0] >= 0)
            & (pixels[:, 0] < VIDEO_WIDTH)
            & (pixels[:, 1] >= 0)
            & (pixels[:, 1] < VIDEO_HEIGHT)
        )
        visible_indices = indices[in_camera]
        states[frame, visible_indices] = side_codes[visible_indices, 0]

print("classified shape (frames, slots):", states.shape)
print("state values:", dict(zip(state_names, np.bincount(states.ravel(), minlength=5))))

## View the interval bars

Each row is one ATD tracker slot. Adjacent frames with the same state are merged into one colored interval. The x-axis is elapsed video time, including the halftime gap.

The diagnostic bars at the end of the notebook show velocity and goalkeeper half for trackers 24 and 46.

In [ ]:
def state_runs(values):
    """Return (start_frame, duration, state) runs for a one-dimensional state series."""
    changes = np.flatnonzero(values[1:] != values[:-1]) + 1
    boundaries = np.r_[0, changes, len(values)]
    return [
        (int(start), int(end - start), int(values[start]))
        for start, end in zip(boundaries[:-1], boundaries[1:])
        if end > start
    ]


fig_height = max(8, 0.24 * len(slots) + 2.5)
fig, ax = plt.subplots(figsize=(16, fig_height), constrained_layout=True)
for slot_index, slot in enumerate(slots):
    y = len(slots) - slot_index - 1
    for start_frame, duration, state in state_runs(states[:, slot_index]):
        ax.broken_barh(
            [(start_frame / FPS / 60, duration / FPS / 60)],
            (y - 0.38, 0.76),
            facecolors=state_colors[state],
            linewidth=0,
        )

ax.set_xlim(0, N_FRAMES / FPS / 60)
ax.set_ylim(-0.7, len(slots) - 0.3)
ax.set_yticks(np.arange(len(slots)))
ax.set_yticklabels([
    "{}  ·  {}  ({})".format(slot["id"], slot["name"], teams[slot["team_id"]])
    for slot in reversed(slots)
])
ax.set_xlabel("Elapsed video time (minutes)")
ax.set_title("BAR-ATH tactical tracker slots: on-pitch and camera visibility")
ax.grid(axis="x", color="#cccccc", alpha=0.45)
ax.set_axisbelow(True)
ax.legend(
    handles=[Patch(facecolor=color, label=name) for name, color in zip(state_names, state_colors)],
    loc="upper center", bbox_to_anchor=(0.5, -0.035), ncol=5, frameon=False,
)
plt.show()

## Percentages for each tracker label

The five percentages for each label sum to 100%. This makes absences and camera cuts visible in the summary as well as in the bars.

In [ ]:
counts = np.stack([
    np.count_nonzero(states == state, axis=0)
    for state in range(len(state_names))
], axis=1)
percentages = counts / N_FRAMES * 100.0
summary = pd.DataFrame(
    percentages,
    columns=state_names,
    index=["{} · {}".format(slot["id"], slot["name"]) for slot in slots],
)
summary.index.name = "Tracker label"
summary.style.format("{:.2f}%").background_gradient(cmap="Blues", axis=None)

## Speed and goalkeeper-half diagnostics

For trackers 24 and 46, the speed bar runs from black (0 m/s) to white (10 m/s); speeds above 10 m/s are red. Orange indicates missing speed, and yellow indicates off-pitch or not-visible tracking. The second bar is green when the goalkeeper is in their own half and blue when in the opponent's half.

In [ ]:
TRACKER_IDS = ("24", "46")
slot_indices = {slot["id"]: i for i, slot in enumerate(slots)}

team_attack_first_half = {}
for element in metadata_root.iter():
    if local_name(element) == "Team":
        team_id = element.get("id") or element.get("teamId")
        for parameter in element.iter():
            if local_name(parameter) == "ProviderParameter":
                name = next((c.text or "" for c in parameter if local_name(c) == "Name"), "")
                if name.strip() == "attack_direction_first_half":
                    team_attack_first_half[team_id] = next(
                        (c.text or "" for c in parameter if local_name(c) == "Value"), ""
                    ).strip()

second_half_start = GAMES[GAME]["halves"][2][0] - GAMES[GAME]["offsets"][2]
second_half = np.arange(N_FRAMES) >= second_half_start
bars = []
labels = []
for tracker_id in TRACKER_IDS:
    i = slot_indices[tracker_id]
    slot = slots[i]
    speed = player_speeds[:, i]
    on_pitch = states[:, i] != NOT_ON_PITCH
    visible = np.isin(states[:, i], [HOME_VISIBLE, AWAY_VISIBLE])

    speed_bar = np.full((N_FRAMES, 3), [255, 255, 0], dtype=np.uint8)
    over_10 = np.isfinite(speed) & (speed > 10)
    speed_bar[over_10] = [255, 0, 0]
    missing_speed = visible & on_pitch & ~np.isfinite(speed)
    speed_bar[missing_speed] = [255, 165, 0]
    gray_frames = visible & on_pitch & np.isfinite(speed) & ~over_10
    gray = (np.clip(speed / 2, 0, 1) * 255).astype(np.uint8)
    speed_bar[gray_frames] = np.repeat(gray[gray_frames, None], 3, axis=1)
    bars.append(speed_bar)
    labels.append("{} speed".format(tracker_id))

    direction = team_attack_first_half[slot["team_id"]]
    own_half_is_left = np.where(
        second_half, direction == "right_to_left", direction == "left_to_right"
    )
    x = tracking_positions[:, i, 0]
    own_half = np.where(own_half_is_left, x <= 0.5, x >= 0.5)
    half_bar = np.full((N_FRAMES, 3), [255, 255, 0], dtype=np.uint8)
    half_bar[on_pitch & own_half] = [0, 160, 0]
    half_bar[on_pitch & ~own_half] = [0, 0, 255]
    bars.append(half_bar)
    labels.append("{} own half".format(tracker_id))

fig, ax = plt.subplots(figsize=(16, 4), constrained_layout=True)
ax.imshow(
    np.stack(bars), aspect="auto", interpolation="nearest", origin="upper",
    extent=(0, N_FRAMES / FPS / 60, len(labels) - 0.5, -0.5),
)
ax.set_yticks(np.arange(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel("Elapsed video time (minutes)")
ax.set_title("Tracker speed and goalkeeper half")
ax.grid(axis="x", color="#cccccc", alpha=0.45)
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
fig.colorbar(
    ScalarMappable(norm=Normalize(0, 10), cmap="gray"),
    ax=ax, label="Speed (m/s): black = 0, white = 10",
)
ax.legend(handles=[
    Patch(facecolor="#ff0000", label="> 10 m/s"),
    Patch(facecolor="#ffa500", label="Speed missing"),
    Patch(facecolor="#ffff00", label="Off pitch / not visible"),
    Patch(facecolor="#00a000", label="Own half"),
    Patch(facecolor="#0000ff", label="Opponent half"),
], loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False)
plt.show()

In [ ]:
FRAME_START, FRAME_END = 30000, 60000
frame_slice = slice(FRAME_START, FRAME_END + 1)
frame_count = FRAME_END - FRAME_START + 1
bars = []
labels = []
for tracker_id in TRACKER_IDS:
    i = slot_indices[tracker_id]
    slot = slots[i]
    speed = player_speeds[frame_slice, i]
    on_pitch = states[frame_slice, i] != NOT_ON_PITCH
    visible = np.isin(states[frame_slice, i], [HOME_VISIBLE, AWAY_VISIBLE])

    speed_bar = np.full((frame_count, 3), [255, 255, 0], dtype=np.uint8)
    over_10 = np.isfinite(speed) & (speed > 10)
    speed_bar[over_10] = [255, 0, 0]
    missing_speed = visible & on_pitch & ~np.isfinite(speed)
    speed_bar[missing_speed] = [255, 165, 0]
    gray_frames = visible & on_pitch & np.isfinite(speed) & ~over_10
    gray = (np.clip(speed / 2, 0, 1) * 255).astype(np.uint8)
    speed_bar[gray_frames] = np.repeat(gray[gray_frames, None], 3, axis=1)
    bars.append(speed_bar)
    labels.append("{} speed".format(tracker_id))

    direction = team_attack_first_half[slot["team_id"]]
    own_half_is_left = np.where(
        second_half[frame_slice], direction == "right_to_left", direction == "left_to_right"
    )
    x = tracking_positions[frame_slice, i, 0]
    own_half = np.where(own_half_is_left, x <= 0.5, x >= 0.5)
    half_bar = np.full((frame_count, 3), [255, 255, 0], dtype=np.uint8)
    half_bar[on_pitch & own_half] = [0, 160, 0]
    half_bar[on_pitch & ~own_half] = [0, 0, 255]
    bars.append(half_bar)
    labels.append("{} own half".format(tracker_id))

fig, ax = plt.subplots(figsize=(16, 4), constrained_layout=True)
ax.imshow(
    np.stack(bars), aspect="auto", interpolation="nearest", origin="upper",
    extent=(FRAME_START / FPS / 60, (FRAME_END + 1) / FPS / 60, len(labels) - 0.5, -0.5),
)
ax.set_yticks(np.arange(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel("Elapsed video time (minutes)")
ax.set_title("Tracker speed and goalkeeper half — frames 7500–15000")
ax.grid(axis="x", color="#cccccc", alpha=0.45)
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
fig.colorbar(
    ScalarMappable(norm=Normalize(0, 10), cmap="gray"),
    ax=ax, label="Speed (m/s): black = 0, white = 10",
)
ax.legend(handles=[
    Patch(facecolor="#ff0000", label="> 10 m/s"),
    Patch(facecolor="#ffa500", label="Speed missing"),
    Patch(facecolor="#ffff00", label="Off pitch / not visible"),
    Patch(facecolor="#00a000", label="Own half"),
    Patch(facecolor="#0000ff", label="Opponent half"),
], loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False)
plt.show()

In [ ]:
visible_states = [HOME_VISIBLE, AWAY_VISIBLE]
both_visible = (
    np.isin(states[:, slot_indices["24"]], visible_states)
    & np.isin(states[:, slot_indices["46"]], visible_states)
)
print("Zero-based video frames where both trackers are camera-visible:")
print(np.flatnonzero(both_visible))

## Derive team goalkeeper tracks

Run this after the classification cells above. It reuses `states`, `tracking_positions`, `player_speeds`, the tracker/team metadata, and the existing second-half boundary. A keeper is assigned to the team defending that physical pitch half. If both source trackers overlap on one pitch half, raw ATD speed selects a source only when exactly one tracker has any finite speed in that contiguous overlap; the chosen source is carried across speed-missing frames within that overlap. Cases with zero or two speed-bearing sources remain unresolved rather than guessed. The original source tracks are not changed. Results are saved under `data/derived/` as a CSV and compressed NPZ.

In [ ]:
# Build two derived, team-specific goalkeeper tracks without changing ATD source slots.
required_trackers = ("24", "46")
missing_trackers = [track_id for track_id in required_trackers if track_id not in slot_indices]
if missing_trackers:
    raise ValueError("Missing goalkeeper tracker slots: {}".format(missing_trackers))
if len(states) != N_FRAMES or len(tracking_positions) != N_FRAMES or len(player_speeds) != N_FRAMES:
    raise ValueError("Tracking arrays do not match the video frame count")

keeper_slot_indices = [slot_indices[track_id] for track_id in required_trackers]
keeper_positions = tracking_positions[:, keeper_slot_indices, :]
keeper_speeds = player_speeds[:, keeper_slot_indices]
keeper_visible = np.column_stack([
    np.isin(states[:, i], [HOME_VISIBLE, AWAY_VISIBLE]) for i in keeper_slot_indices
])
keeper_pitch_half = np.where(keeper_positions[:, :, 0] <= 0.5, 0, 1).astype(np.int8)

# The keeper defends the opposite pitch half from the direction their team attacks.
side_to_team_id = {team_side[team_id]: team_id for team_id in team_ids}
team_target_pitch_half = {}
for side in ("home", "away"):
    team_id = side_to_team_id[side]
    direction = team_attack_first_half.get(team_id)
    if direction not in ("left_to_right", "right_to_left"):
        raise ValueError("Unknown first-half attack direction for {}: {!r}".format(side, direction))
    first_half_defending_half = 0 if direction == "left_to_right" else 1
    team_target_pitch_half[side] = np.where(
        second_half, 1 - first_half_defending_half, first_half_defending_half
    ).astype(np.int8)

team_names_by_side = {team_side[team_id]: teams[team_id] for team_id in team_ids}
goalkeeper_tracks_derived = {
    side: {
        "position": np.full((N_FRAMES, 2), np.nan, dtype=np.float32),
        "speed": np.full(N_FRAMES, np.nan, dtype=np.float32),
        "source_track_id": np.full(N_FRAMES, "", dtype="<U2"),
        "candidate_track_ids": np.full(N_FRAMES, "", dtype="<U8"),
        "status": np.full(N_FRAMES, "no_visible_tracker", dtype="<U48"),
        "pitch_half": np.full(N_FRAMES, "", dtype="<U4"),
    }
    for side in ("home", "away")
}


def team_for_pitch_half(frame, half_code):
    matches = [
        side for side in ("home", "away")
        if team_target_pitch_half[side][frame] == half_code
    ]
    if len(matches) != 1:
        raise ValueError("Could not uniquely map pitch half at frame {}".format(frame))
    return matches[0]


def assign_goalkeeper(frame, side, source_index, status):
    source_id = required_trackers[source_index]
    track = goalkeeper_tracks_derived[side]
    track["position"][frame] = keeper_positions[frame, source_index]
    track["speed"][frame] = keeper_speeds[frame, source_index]
    track["source_track_id"][frame] = source_id
    track["candidate_track_ids"][frame] = source_id
    track["status"][frame] = status
    track["pitch_half"][frame] = "left" if keeper_pitch_half[frame, source_index] == 0 else "right"


same_half_overlap = (
    keeper_visible[:, 0] & keeper_visible[:, 1]
    & (keeper_pitch_half[:, 0] == keeper_pitch_half[:, 1])
)
for frame in np.flatnonzero(keeper_visible.any(axis=1) & ~same_half_overlap):
    for source_index in np.flatnonzero(keeper_visible[frame]):
        half_code = keeper_pitch_half[frame, source_index]
        side = team_for_pitch_half(frame, half_code)
        assign_goalkeeper(frame, side, source_index, "assigned_by_pitch_half")

# In a same-half overlap, assign the goalkeeper in that half to the tracker
# with speed on each frame; tracker team metadata does not decide the source.
edges = np.flatnonzero(np.diff(np.r_[False, same_half_overlap, False].astype(np.int8)))
for interval_start, interval_end in zip(edges[::2], edges[1::2]):
    speed_present = np.isfinite(keeper_speeds[interval_start:interval_end]).any(axis=0)
    for frame in range(interval_start, interval_end):
        half_code = keeper_pitch_half[frame, 0]
        side = team_for_pitch_half(frame, half_code)
        other_side = "away" if side == "home" else "home"
        goalkeeper_tracks_derived[other_side]["status"][frame] = "no_label_in_defending_half"
        active_sources = np.flatnonzero(np.isfinite(keeper_speeds[frame]))
        if len(active_sources) == 1:
            assign_goalkeeper(frame, side, int(active_sources[0]), "speed_disambiguated")
        elif len(active_sources) == 2:
            track = goalkeeper_tracks_derived[side]
            track["status"][frame] = "unresolved_same_half_both_active"
            track["candidate_track_ids"][frame] = ";".join(required_trackers)
        elif speed_present.sum() == 1:
            source_index = int(np.flatnonzero(speed_present)[0])
            assign_goalkeeper(frame, side, source_index, "speed_inferred_within_overlap")
        else:
            track = goalkeeper_tracks_derived[side]
            track["status"][frame] = "unresolved_same_half_no_speed"
            track["candidate_track_ids"][frame] = ";".join(required_trackers)

output_dir = ROOT / "data" / "derived"
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "{}_tactical_goalkeeper_tracks.csv".format(GAME)
npz_path = output_dir / "{}_tactical_goalkeeper_tracks.npz".format(GAME)

frames = np.arange(N_FRAMES, dtype=np.int64)
csv_parts = []
for side in ("home", "away"):
    track = goalkeeper_tracks_derived[side]
    csv_parts.append(pd.DataFrame({
        "frame": frames,
        "time_seconds": frames / FPS,
        "team_side": side,
        "team_name": team_names_by_side[side],
        "x": track["position"][:, 0],
        "y": track["position"][:, 1],
        "speed_m_s": track["speed"],
        "source_track_id": track["source_track_id"],
        "candidate_track_ids": track["candidate_track_ids"],
        "pitch_half": track["pitch_half"],
        "status": track["status"],
    }))
pd.concat(csv_parts, ignore_index=True).to_csv(csv_path, index=False)

np.savez_compressed(
    npz_path,
    frames=frames,
    fps=FPS,
    team_sides=np.asarray(["home", "away"]),
    team_names=np.asarray([team_names_by_side["home"], team_names_by_side["away"]]),
    positions=np.stack([
        goalkeeper_tracks_derived[side]["position"] for side in ("home", "away")
    ], axis=1),
    speeds=np.stack([
        goalkeeper_tracks_derived[side]["speed"] for side in ("home", "away")
    ], axis=1),
    source_track_ids=np.stack([
        goalkeeper_tracks_derived[side]["source_track_id"] for side in ("home", "away")
    ], axis=1),
    candidate_track_ids=np.stack([
        goalkeeper_tracks_derived[side]["candidate_track_ids"] for side in ("home", "away")
    ], axis=1),
    pitch_halves=np.stack([
        goalkeeper_tracks_derived[side]["pitch_half"] for side in ("home", "away")
    ], axis=1),
    statuses=np.stack([
        goalkeeper_tracks_derived[side]["status"] for side in ("home", "away")
    ], axis=1),
)

print("Saved derived goalkeeper tracks:")
print(" CSV:", csv_path)
print(" NPZ:", npz_path)
for side in ("home", "away"):
    status, count = np.unique(goalkeeper_tracks_derived[side]["status"], return_counts=True)
    print(side, team_names_by_side[side], dict(zip(status.tolist(), count.tolist())))
